1. Retrieve-only contextual compression (no generation)
2. Conversational RAG with memory, **without** an agent
3. That same RAG pipeline wrapped as a **tool inside an agent**
4. `MultiQueryRetriever`-based retrieval + generation, with sources

Migration cheat-sheet
Legacy piece	v1 (langchain==1.3.11)
langchain.chat_models.ChatOpenAI / langchain.embeddings.OpenAIEmbeddings	langchain_openai.ChatOpenAI / langchain_openai.OpenAIEmbeddings
langchain.vectorstores.FAISS	langchain_community.vectorstores.FAISS
langchain.text_splitter.CharacterTextSplitter	langchain_text_splitters.CharacterTextSplitter
langchain.schema.Document	langchain_core.documents.Document
langchain.retrievers.ContextualCompressionRetriever / .document_compressors.LLMChainExtractor	langchain_classic.retrievers.ContextualCompressionRetriever / langchain_classic.retrievers.document_compressors.LLMChainExtractor (verified import — pip install langchain-classic)
langchain.retrievers.multi_query.MultiQueryRetriever	langchain_classic.retrievers.multi_query.MultiQueryRetriever (verified import)
retriever.get_relevant_documents(query)	retriever.invoke(query)
ConversationalRetrievalChain + ConversationBufferMemory	Deprecated, moved to langchain-classic. Recommended v1 replacement: explicit retrieval + an explicit chat-history list (shown below)
RetrievalQA.from_chain_type(...)	Deprecated. Recommended v1 replacement: retriever.invoke(query) followed by a direct llm.invoke(...) call over the retrieved context
initialize_agent(..., AgentType.CONVERSATIONAL_REACT_DESCRIPTION) + StructuredTool	create_agent(model, tools, checkpointer=...) + @tool


Reference: https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent

In [1]:
# 📦 Required installs
#!pip install "langchain==1.3.11" langchain-openai langchain-community langchain-classic langchain-text-splitters faiss-cpu python-dotenv langgraph

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

import os
from dotenv import load_dotenv

# 2️⃣ Load API keys
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = "Openai_api_Key"

# 1. Load the text file
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 2. Split the text into chunks
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(raw_text)
documents = [Document(page_content=chunk) for chunk in chunks]

# 3. Create vector store
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents, embedding=embedding)

# 4. Setup LLM
# NOTE: the original notebook relied on the default model (gpt-3.5-turbo era),
# which has since been retired — pin an explicit, currently-supported model.
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

C:\Users\KAVITHA\AppData\Local\Temp\ipykernel_13260\3017802901.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


1) Retrieve-only contextual compression
ContextualCompressionRetriever and LLMChainExtractor still work exactly as before — they now live in langchain-classic rather than langchain. The only functional change is swapping the deprecated .get_relevant_documents() for .invoke().

In [2]:
## 1) Retrieve-only contextual compression
# Get retrieved docs --> it does not generate an answer, it only compresses/filters them
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Compressor using LLM
compressor = LLMChainExtractor.from_llm(llm)

# Create compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)

# Run retrieval (.invoke replaces the deprecated .get_relevant_documents())
print("🔹 Contextual Compression Results:")
results = compression_retriever.invoke("Who created LangChain?")
for doc in results:
    print("-", doc.page_content)

🔹 Contextual Compression Results:
- LangChain was created by Harrison Chase.


2) Conversational RAG with memory — without agent mode
ConversationalRetrievalChain and ConversationBufferMemory are deprecated (moved to langchain-classic). Since this scenario is specifically about not using an agent framework, the v1-native way to do it is the same thing LangChain's own migration guide recommends: skip the chain class entirely and write the retrieval + history-aware prompt explicitly. It's a few more lines, but there's no chain black-box and no memory object to wire up.

In [3]:
## 2) Conversational RAG with memory — without agent mode
# Integration with a conversational RAG "chain" — without agent mode
# (ConversationalRetrievalChain is deprecated; this is the LangChain-recommended
#  manual replacement: explicit retrieval + an explicit chat-history list)

chat_history: list[tuple[str, str]] = []  # [(question, answer), ...]

def rag_chain_fn(question: str) -> str:
    docs = compression_retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    history_text = "\n".join(f"Q: {q}\nA: {a}" for q, a in chat_history)
    prompt = (
        "Answer the question using the conversation history and context below.\n\n"
        f"Conversation history:\n{history_text}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    answer = llm.invoke(prompt).content
    chat_history.append((question, answer))
    return answer

print("🔹 ConversationalRetrievalChain (no agent):")
print(rag_chain_fn("What is LangChain?"))
print(rag_chain_fn("Who created it?"))

🔹 ConversationalRetrievalChain (no agent):
LangChain is a framework for building applications with large language models (LLMs). It was created by Harrison Chase and supports features such as retrieval-augmented generation (RAG), agents, memory, and tools. LangChain is commonly used in applications like chatbots, document question answering, and AI workflows.
LangChain was created by Harrison Chase.


3) Integration into an agent (with a tool)
The original code wrapped the chain in a StructuredTool and used initialize_agent(..., AgentType.CONVERSATIONAL_REACT_DESCRIPTION), manually re-injecting history via "chat_history": [memory.chat_memory.messages] — which actually has a bug: it wraps the whole message list inside another list, one level too deep.

create_agent needs none of that: give it a checkpointer and a thread_id, and it tracks conversation state itself, so there's no chat-history plumbing left to get wrong.

In [4]:
## 3) Integration into an agent (with a tool)
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def RAG_Tool(question: str) -> str:
    """Answer LangChain-related questions with context."""
    docs = compression_retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)
    answer = llm.invoke(
        f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
    )
    return answer.content

checkpointer = InMemorySaver()
thread_config = {"configurable": {"thread_id": "compression-demo-1"}}

agent = create_agent(
    model=llm,
    tools=[RAG_Tool],
    checkpointer=checkpointer,
)

print("\n🔹 Agent Conversation:")
res1 = agent.invoke({"messages": [{"role": "user", "content": "What is LangChain?"}]}, thread_config)
print(res1["messages"][-1].content)

res2 = agent.invoke({"messages": [{"role": "user", "content": "Who created it?"}]}, thread_config)
print(res2["messages"][-1].content)


🔹 Agent Conversation:
LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides tools and components to help developers create applications that leverage the capabilities of these models effectively. If you want, I can provide more detailed information about its features and use cases.
LangChain was created by Harrison Chase. If you want to know more about the creator or the history of LangChain, feel free to ask!


4) MultiQueryRetriever
MultiQueryRetriever is a retrieval-only utility (still verified-importable from langchain_classic.retrievers.multi_query, same .from_llm(retriever=..., llm=...) signature as before). RetrievalQA — the generation chain the original notebook used on top of it — is deprecated, so generation here is done the same explicit way as the rest of this notebook: retrieve, then call the LLM directly over the retrieved context.

In [5]:
## 4) MultiQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# MultiQueryRetriever (still available via langchain-classic)
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Retrieve + generate directly (replaces the deprecated RetrievalQA chain)
query = "Tell me about LangChain creator and features."
source_docs = multi_query_retriever.invoke(query)
context = "\n".join(doc.page_content for doc in source_docs)

answer = llm.invoke(
    f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"
).content

print("\n🔹 RAG Pipeline (MultiQueryRetriever):")
print("Answer:", answer)

print("\nSources:")
for doc in source_docs:
    print("-", doc.page_content[:200])  # print first 200 chars of each doc


🔹 RAG Pipeline (MultiQueryRetriever):
Answer: LangChain was created by Harrison Chase. Its features include support for RAG (Retrieval-Augmented Generation), agents, memory, tools, and more.

Sources:
- LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&


Summary
ContextualCompressionRetriever, LLMChainExtractor, and MultiQueryRetriever are all still usable as-is — only their import path changed, to langchain_classic.
ConversationalRetrievalChain and RetrievalQA are deprecated; both are replaced here with an explicit "retrieve, then call the LLM" pattern, which also removes two real bugs from the original notebook (chat history that was never actually passed to the chain, and a chat-history list nested one level too deep).
initialize_agent + AgentType → create_agent + checkpointer, same as the rest of this series of rewrites.
For more detail, see: https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent